In Chapter 3, you saw how to fine-tune a model for text classification. In this chapter, we will tackle the following common language tasks that are essential for working with both traditional NLP models and modern LLMs:

- Token classification
- Masked language modeling (like BERT)
- Summarization
- Translation
- Causal language modeling pretraining (like GPT-2)
- Question answering

These fundamental tasks form the foundation of how Large Language Models (LLMs) work and understanding them is crucial for effectively working with today’s most advanced language models.

To do this, you’ll need to leverage everything you learned about the Trainer API and the 🤗 Accelerate library in Chapter 3, the 🤗 Datasets library in Chapter 5, and the 🤗 Tokenizers library in Chapter 6. We’ll also upload our results to the Model Hub, like we did in Chapter 4, so this is really the chapter where everything comes together!

Each section can be read independently and will show you how to train a model with the Trainer API or with your own training loop, using 🤗 Accelerate. Feel free to skip either part and focus on the one that interests you the most: the Trainer API is great for fine-tuning or training your model without worrying about what’s going on behind the scenes, while the training loop with Accelerate will let you customize any part you want more easily.

### Token classification

The first application we’ll explore is token classification. This generic task encompasses any problem that can be formulated as “attributing a label to each token in a sentence,” such as:

- Named entity recognition (NER): Find the entities (such as persons, locations, or organizations) in a sentence. This can be formulated as attributing a label to each token by having one class per entity and one class for “no entity.”
- Part-of-speech tagging (POS): Mark each word in a sentence as corresponding to a particular part of speech (such as noun, verb, adjective, etc.).
- Chunking: Find the tokens that belong to the same entity. This task (which can be combined with POS or NER) can be formulated as attributing one label (usually B-) to any tokens that are at the beginning of a chunk, another label (usually I-) to tokens that are inside a chunk, and a third label (usually O) to tokens that don’t belong to any chunk.

In [3]:
import numpy as np
from datasets import load_dataset

raw_datasets = load_dataset(
    "eriktks/conll2003",
    "default",
    revision="refs/convert/parquet",)

C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\huggingface_cache\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlin

In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [5]:
raw_datasets['train'][0]['tokens']

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']

In [6]:
raw_datasets["train"][0]["ner_tags"]

[3, 0, 7, 0, 0, 0, 7, 0, 0]

In [7]:
ner_feature = raw_datasets['train'].features['ner_tags']
ner_feature

List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))

In [8]:
label_names = ner_feature.feature.names

In [9]:
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [10]:
words = raw_datasets["train"][0]["tokens"]
labels = raw_datasets["train"][0]["ner_tags"]
line1 = ""
line2 = ""
for word, label in zip(words, labels):
    full_label = label_names[label]
    max_length = max(len(word), len(full_label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += full_label + " " * (max_length - len(full_label) + 1)

print(line1)
print(line2)

EU    rejects German call to boycott British lamb . 
B-ORG O       B-MISC O    O  O       B-MISC  O    O 


In [11]:
tag_column = "pos_tags"  # Change to "chunk_tags" for chunking

label_names = raw_datasets["train"].features[tag_column].feature.names

for i in [0, 1]:
    words = raw_datasets["train"][i]["tokens"]
    labels = raw_datasets["train"][i][tag_column]

    line1 = ""
    line2 = ""

    for word, label in zip(words, labels):
        full_label = label_names[label]
        max_length = max(len(word), len(full_label))

        line1 += word + " " * (max_length - len(word) + 1)
        line2 += full_label + " " * (max_length - len(full_label) + 1)

    print(line1)
    print(line2)
    print()

EU  rejects German call to boycott British lamb . 
NNP VBZ     JJ     NN   TO VB      JJ      NN   . 

Peter Blackburn 
NNP   NNP       



In [12]:
tag_column = "chunk_tags"  # Change to "chunk_tags" for chunking

label_names = raw_datasets["train"].features[tag_column].feature.names

for i in [0, 1]:
    words = raw_datasets["train"][i]["tokens"]
    labels = raw_datasets["train"][i][tag_column]

    line1 = ""
    line2 = ""

    for word, label in zip(words, labels):
        full_label = label_names[label]
        max_length = max(len(word), len(full_label))

        line1 += word + " " * (max_length - len(word) + 1)
        line2 += full_label + " " * (max_length - len(full_label) + 1)

    print(line1)
    print(line2)
    print()

EU   rejects German call to   boycott British lamb . 
B-NP B-VP    B-NP   I-NP B-VP I-VP    B-NP    I-NP O 

Peter Blackburn 
B-NP  I-NP      



In [13]:
from transformers import AutoTokenizer

model_checkpoint = 'bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [14]:
tokenizer.is_fast

True

In [16]:
inputs = tokenizer(raw_datasets['train'][0]['tokens'], is_split_into_words=True)
inputs.tokens()

['[CLS]',
 'EU',
 'rejects',
 'German',
 'call',
 'to',
 'boycott',
 'British',
 'la',
 '##mb',
 '.',
 '[SEP]']

With a tiny bit of work, we can then expand our label list to match the tokens. The first rule we’ll apply is that special tokens get a label of -100. This is because by default -100 is an index that is ignored in the loss function we will use (cross entropy). Then, each token gets the same label as the token that started the word it’s inside, since they are part of the same entity. For tokens inside a word but not at the beginning, we replace the B- with I- (since the token does not begin the entity):

In [17]:
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None

    for word_id in word_ids:

        if word_id != current_word:
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)

        elif word_id is None:
            new_labels.append(-100)

        else:
            label = labels[word_id]

            if label % 2 == 1:
                label += 1

            new_labels.append(label)

    return new_labels

In [18]:
labels = raw_datasets["train"][0]["ner_tags"]
word_ids = inputs.word_ids()
print(labels)
print(align_labels_with_tokens(labels, word_ids))

[3, 0, 7, 0, 0, 0, 7, 0, 0]
[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]


In [19]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [20]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

Map: 100%|██████████| 3453/3453 [00:00<00:00, 43678.41 examples/s]


In [21]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [22]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(2)])
batch["labels"]

tensor([[-100,    3,    0,    7,    0,    0,    0,    7,    0,    0,    0, -100],
        [-100,    1,    2, -100, -100, -100, -100, -100, -100, -100, -100, -100]])

In [23]:
import evaluate

metric = evaluate.load('seqeval')

In [24]:
labels = raw_datasets["train"][0]["ner_tags"]
labels = [label_names[i] for i in labels]
labels

['B-ADVP', 'O', 'B-INTJ', 'O', 'O', 'O', 'B-INTJ', 'O', 'O']

In [25]:
predictions = labels.copy()
predictions[2] = "O"
metric.compute(predictions=[predictions], references=[labels])

{'ADVP': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(1)},
 'INTJ': {'precision': np.float64(1.0),
  'recall': np.float64(0.5),
  'f1': np.float64(0.6666666666666666),
  'number': np.int64(2)},
 'overall_precision': np.float64(1.0),
 'overall_recall': np.float64(0.6666666666666666),
 'overall_f1': np.float64(0.8),
 'overall_accuracy': 0.8888888888888888}

In [33]:
import numpy as np

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [34]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

In [35]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 832.73it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not o

In [36]:
from huggingface_hub import notebook_login

notebook_login()

In [37]:
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

In [38]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.078506,0.069698,0.905469,0.933356,0.919201,0.980882
2,0.034125,0.067136,0.936128,0.947156,0.941610,0.985283
3,0.021670,0.058374,0.936463,0.950017,0.943191,0.986446


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.01it/s]


TrainOutput(global_step=5268, training_loss=0.07195312956923686, metrics={'train_runtime': 324.557, 'train_samples_per_second': 129.786, 'train_steps_per_second': 16.231, 'total_flos': 920888121858078.0, 'train_loss': 0.07195312956923686, 'epoch': 3.0})

In [40]:
trainer.push_to_hub(commit_message="Training complete")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/alish12313/bert-finetuned-ner/commit/9f51df749f43b851621589c844e939d2b7f814f2', commit_message='Training complete', commit_description='', oid='9f51df749f43b851621589c844e939d2b7f814f2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/alish12313/bert-finetuned-ner', endpoint='https://huggingface.co', repo_type='model', repo_id='alish12313/bert-finetuned-ner'), pr_revision=None, pr_num=None)

In [41]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=8
)

In [42]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 612.15it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not o

In [43]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [44]:
from accelerate import Accelerator

accelerator = Accelerator()

model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [45]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    'linear',
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

In [53]:
def postprocess(predictions, labels):
    predictions = predictions.detach().cpu().clone().numpy()
    labels = labels.detach().cpu().clone().numpy()

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return true_labels, true_predictions

In [56]:
from tqdm.auto import tqdm
import torch

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_training_steps):
    model.train()

    for batch in train_dataloader:
        y_pred = model(**batch)
        loss = y_pred.loss
        accelerator.backward(loss)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    model.eval()
    for batch in eval_dataloader:
        with torch.inference_mode():
            y_pred = model(**batch)

        predictions = y_pred.logits.argmax(dim=-1)
        labels = batch["labels"]

        predictions = accelerator.pad_across_processes(predictions, dim=1, pad_index=-100)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)

        predictions_gathered = accelerator.gather(predictions)
        labels_gathered = accelerator.gather(labels)

        true_predictions, true_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=true_predictions, references=true_labels)

        results = metric.compute()
        print(
            f"epoch {epoch}:",
            {
                key: results[f"overall_{key}"]
                for key in ["precision", "recall", "f1", "accuracy"]
            },
        )

        accelerator.wait_for_everyone()
        unwrapped_model = accelerator.unwrap_model(model)

 33%|███▎      | 1755/5268 [02:02<04:00, 14.62it/s]

epoch 0: {'precision': np.float64(0.9615384615384616), 'recall': np.float64(0.9259259259259259), 'f1': np.float64(0.9433962264150944), 'accuracy': 0.9616858237547893}
epoch 0: {'precision': np.float64(0.9565217391304348), 'recall': np.float64(0.9565217391304348), 'f1': np.float64(0.9565217391304348), 'accuracy': 0.988}
epoch 0: {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'accuracy': 1.0}
epoch 0: {'precision': np.float64(0.9743589743589743), 'recall': np.float64(0.95), 'f1': np.float64(0.9620253164556962), 'accuracy': 0.9734513274336283}
epoch 0: {'precision': np.float64(0.8333333333333334), 'recall': np.float64(0.7142857142857143), 'f1': np.float64(0.7692307692307692), 'accuracy': 0.9770114942528736}
epoch 0: {'precision': np.float64(0.8571428571428571), 'recall': np.float64(0.8571428571428571), 'f1': np.float64(0.8571428571428571), 'accuracy': 0.8666666666666667}
epoch 0: {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1

 33%|███▎      | 1757/5268 [02:11<1:16:45,  1.31s/it]

epoch 0: {'precision': np.float64(0.9629629629629629), 'recall': np.float64(0.896551724137931), 'f1': np.float64(0.9285714285714286), 'accuracy': 0.9852216748768473}
epoch 0: {'precision': np.float64(0.9523809523809523), 'recall': np.float64(0.9090909090909091), 'f1': np.float64(0.9302325581395349), 'accuracy': 0.9808612440191388}
epoch 0: {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'accuracy': 1.0}
epoch 0: {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'accuracy': 1.0}
epoch 0: {'precision': np.float64(0.7777777777777778), 'recall': np.float64(0.6363636363636364), 'f1': np.float64(0.7000000000000001), 'accuracy': 0.9356435643564357}
epoch 0: {'precision': np.float64(0.8636363636363636), 'recall': np.float64(0.8636363636363636), 'f1': np.float64(0.8636363636363636), 'accuracy': 0.985781990521327}
epoch 0: {'precision': np.float64(0.875), 'recall': np.float64(0.7777777777777778), 'f1': np.float64(0.823529411764706),

 67%|██████▋   | 3511/5268 [04:17<02:45, 10.61it/s]  

...

epoch 3: {'precision': np.float64(0.9615384615384616), 'recall': np.float64(0.9259259259259259), 'f1': np.float64(0.9433962264150944), 'accuracy': 0.9616858237547893}
epoch 3: {'precision': np.float64(0.9565217391304348), 'recall': np.float64(0.9565217391304348), 'f1': np.float64(0.9565217391304348), 'accuracy': 0.988}
epoch 3: {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'accuracy': 1.0}
epoch 3: {'precision': np.float64(0.9743589743589743), 'recall': np.float64(0.95), 'f1': np.float64(0.9620253164556962), 'accuracy': 0.9734513274336283}
epoch 3: {'precision': np.float64(0.8333333333333334), 'recall': np.float64(0.7142857142857143), 'f1': np.float64(0.7692307692307692), 'accuracy': 0.9770114942528736}
epoch 3: {'precision': np.float64(0.8571428571428571), 'recall': np.float64(0.8571428571428571), 'f1': np.float64(0.8571428571428571), 'accuracy': 0.8666666666666667}
epoch 3: {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1

7025it [08:55,  1.34s/it]

epoch 3: {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'accuracy': 1.0}
epoch 3: {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'accuracy': 1.0}
epoch 3: {'precision': np.float64(0.7777777777777778), 'recall': np.float64(0.6363636363636364), 'f1': np.float64(0.7000000000000001), 'accuracy': 0.9356435643564357}
epoch 3: {'precision': np.float64(0.8636363636363636), 'recall': np.float64(0.8636363636363636), 'f1': np.float64(0.8636363636363636), 'accuracy': 0.985781990521327}
epoch 3: {'precision': np.float64(0.875), 'recall': np.float64(0.7777777777777778), 'f1': np.float64(0.823529411764706), 'accuracy': 0.9949238578680203}
epoch 3: {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'accuracy': 1.0}
epoch 3: {'precision': np.float64(0.5), 'recall': np.float64(0.5), 'f1': np.float64(0.5), 'accuracy': 0.9607843137254902}


7595it [09:35, 14.32it/s]

KeyboardInterrupt: 